# EYES-DEFY-ANEMIA -- Step 1: Measurement Harness (Transformers)

Pooled out-of-fold repeated stratified cross-validation for the 6 clean-data transformer combos
(`swin_t`, `vit_b_16`, `vit_l_16` x both tissue types). Same measurement harness as the CNN run --
**no model is changed and no intervention is applied.**

**Companion notebook:** `step1-cv-harness-cnn.ipynb` covers the 12 CNN combos separately. Run both,
then merge their downloaded `outputs/` locally before running `aggregate_baseline.py` to get the
full 18-combo Step 1 baseline. This notebook alone produces a 6-combo (transformer-only) baseline.

**Why the same problem applies here too.** The single 70/15/15 split leaves 14 India validation
patients (10 anemic / 4 healthy), so India AUC is computed over 10x4 = 40 discordant pairs --
a 95% CI half-width of roughly +/-0.27, regardless of which architecture produced the scores. The
pooled design here raises that to the same **1,311 India pairs** (57x23) as the CNN run, since both
draw on the identical 184-patient pool and fold design.

**The 33-patient test split is sealed** and is asserted absent from every fold, exactly as in the
CNN notebook. It is spent exactly once, in Step 6.

**Runtime note -- read before launching.** 6 combos x 25 fits (5 folds x 5 repeats), and this is
the heaviest architecture roster in the project: `vit_l_16` alone is 304.33M frozen backbone
parameters, so even though only the head trains, every fit still pays the full forward-pass cost
through the whole network, 25 times. This notebook is very likely the single most expensive run in
the project to date. `sync_outputs()` runs after every combo, so an interrupted session still
yields a downloadable zip of everything completed so far. **If the session looks tight on time,
add `--repeats 3` to the training cells** (documented budget fallback -- CI half-width will be
wider than the 5-repeat design target, but the run stays honest and comparable).

## Setup

In [1]:
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

CUDA available: True
GPU: Tesla T4


In [2]:
# rm -rf first so a re-run within the same kernel session stays idempotent
# instead of nesting a second clone inside the first (this bit an earlier
# version of this notebook during interactive debugging).
!rm -rf eyes-defy-anemia
!git clone https://github.com/manivafapour/eyes-defy-anemia.git
%cd eyes-defy-anemia

Cloning into 'eyes-defy-anemia'...
remote: Enumerating objects: 940, done.
remote: Counting objects: 100% (621/621), done.
remote: Compressing objects: 100% (415/415), done.
remote: Total 940 (delta 274), reused 531 (delta 203), pack-reused 319 (from 1)
Receiving objects: 100% (940/940), 76.25 MiB | 31.11 MiB/s, done.
Resolving deltas: 100% (437/437), done.
/kaggle/working/eyes-defy-anemia


In [3]:
# Diagnostic only -- kept for visibility in the saved run log. The path used
# below is already confirmed correct, so this cell doesn't gate anything, but
# it's cheap and makes a future path change easy to spot in the output.
import os

for name in os.listdir("/kaggle/input"):
    path = f"/kaggle/input/{name}"
    print(name, "->", os.listdir(path))

datasets -> ['manivafapour33']


In [4]:
# Only packages actually missing from Kaggle's base image. Deliberately NOT
# `pip install -r requirements.txt` -- that file is pinned to the local
# Windows/CUDA 13.0 build and would try to reinstall Kaggle's own correctly
# configured GPU PyTorch with an incompatible build.
!pip install -q optuna albumentations

## Data

In [5]:
import shutil
from pathlib import Path

# TODO: verify against cell 4's /kaggle/input listing before running -- this is a best-guess
# default following the established manivafapour33/<slug> pattern, NOT yet confirmed for the
# new clean-data dataset (see classification/.project_memory/kaggle/01_kaggle_notes.md).
SRC_DIR = Path("/kaggle/input/datasets/manivafapour33/processed-dataset-clean")
DST_DIR = Path("classification/data/processed")

# Clear the destination first so this cell is fully idempotent -- a re-run (or
# the two messy copy attempts from the earlier notebook version) never leaves
# stale or duplicated content behind. DST_DIR always ends up as an exact,
# deterministic copy of SRC_DIR, nothing more.
shutil.rmtree(DST_DIR, ignore_errors=True)
DST_DIR.mkdir(parents=True, exist_ok=True)

for item in SRC_DIR.iterdir():
    dest = DST_DIR / item.name
    if item.is_dir():
        shutil.copytree(item, dest)
    else:
        shutil.copy2(item, dest)

print("classification/data/processed now contains:")
for sub in sorted(DST_DIR.iterdir()):
    if sub.is_dir():
        n_files = sum(1 for f in sub.rglob("*") if f.is_file())
        print(f"  {sub.name}/  ({n_files} files)")
    else:
        print(f"  {sub.name}")

classification/data/processed now contains:
  extraction_log.csv
  images/  (428 files)
  metadata.csv
  splits.csv


## Structural verification -- checkpoints 1-4 and 6

Identical check to the CNN notebook -- the fold geometry it verifies (test-set isolation, partition
integrity, rare-cell coverage, pair counts, reproducibility) depends only on the 184-patient pool and
the fold design, not on which architecture will be trained through it. Trains nothing; runs in seconds.

**This cell gates the run.** A non-zero exit means do not proceed to training.

In [6]:
!python classification/step1_cv_harness/validate_harness.py


=== Structural verification: tissue_type=palpebral (k=5, repeats=5, seed=42) ===
  [PASS] 1. test-set isolation: 0 of 33 held-out test patients appear in any fold
  [PASS] 2. partition integrity: each of 5 repeats partitions all 184 pool patients exactly once
  [PASS] 3. rare-cell coverage: min India_0 per outer val fold = 4 (at r0f0), min Italy_1 per outer val fold = 4 (at r0f0)
  [PASS] 4. pair counts (palpebral): pool n=184 cells={'India_0': 23, 'India_1': 57, 'Italy_0': 84, 'Italy_1': 20} | India 57x23=1311 pairs (32.8x the single split's 40) | Italy 20x84=1680 pairs (28.0x the single split's 60)
  [PASS] 6. fold reproducibility: two independent build_folds(seed=42) calls produced identical assignments
  --> ALL STRUCTURAL CHECKS PASSED

=== Structural verification: tissue_type=forniceal_palpebral (k=5, repeats=5, seed=42) ===
  [PASS] 1. test-set isolation: 0 of 33 held-out test patients appear in any fold
  [PASS] 2. partition integrity: each of 5 repeats partitions all 178 pool

In [7]:
# Locked hyperparameters for the 6 transformer combos, read straight out of each combo's own
# v2_clean study summary (never hand-transcribed). No re-tuning happens in Step 1.
!python classification/step1_cv_harness/run_cv_harness.py --list

18 combos discovered (source: each combo's own v2_clean study summary):

  convnext_tiny_forniceal_palpebral_v2_clean
      arch=convnext_tiny        tissue=forniceal_palpebral  lr=0.0773445 wd=0.000479821 dropout=0.5
  convnext_tiny_palpebral_v2_clean
      arch=convnext_tiny        tissue=palpebral            lr=0.000585494 wd=2.00941e-05 dropout=0.2
  densenet121_forniceal_palpebral_v2_clean
      arch=densenet121          tissue=forniceal_palpebral  lr=0.0183628 wd=0.000153747 dropout=0.5
  densenet121_palpebral_v2_clean
      arch=densenet121          tissue=palpebral            lr=0.000293803 wd=2.93754e-06 dropout=0.5
  efficientnet_b0_forniceal_palpebral_v2_clean
      arch=efficientnet_b0      tissue=forniceal_palpebral  lr=0.00132929 wd=0.000711448 dropout=0.2
  efficientnet_b0_palpebral_v2_clean
      arch=efficientnet_b0      tissue=palpebral            lr=0.0314288 wd=4.33528e-06 dropout=0.5
  mobilenet_v3_small_forniceal_palpebral_v2_clean
      arch=mobilenet_v3_small   

## Output syncing

In [8]:
import shutil
from pathlib import Path


def sync_outputs():
    """Consolidate classification/step1_cv_harness/outputs/ into
    /kaggle/working/outputs/ and re-zip to step1_cv_results_vit.zip. Called after
    EVERY combo, not just at the end -- given the runtime note above, whatever has
    completed so far must always be downloadable."""
    results_dir = Path("/kaggle/working/outputs")
    results_dir.mkdir(parents=True, exist_ok=True)
    src = Path("classification/step1_cv_harness/outputs")
    if src.exists():
        shutil.copytree(src, results_dir, dirs_exist_ok=True)
    archive = shutil.make_archive("/kaggle/working/step1_cv_results_vit", "zip", root_dir=str(results_dir))
    n = sum(1 for f in results_dir.rglob("*") if f.is_file())
    print(f"[sync_outputs] {n} files under {results_dir}, zipped to {archive}")


sync_outputs()  # picks up structural_verification.json; confirms the function works before training

[sync_outputs] 1 files under /kaggle/working/outputs, zipped to /kaggle/working/step1_cv_results_vit.zip


## Training -- 6 combos, cheapest architecture first

25 fits per combo (5 folds x 5 repeats), same protocol as the CNN notebook: inner-split early
stopping (never the outer held-out fold), no checkpoints written -- Step 1 needs predictions and
per-fold diagnostics (loss curves, confusion matrices, ROC curves), not weights.

**Budget fallback if the session is tight on time:** add `--repeats 3` to any cell below.

In [9]:
# Step 1 (transformers) -- 1/6: swin_t_palpebral
!python classification/step1_cv_harness/run_cv_harness.py --combo swin_t_palpebral_v2_clean
sync_outputs()


Step 1 harness: swin_t_palpebral_v2_clean
  device=cuda  arch=swin_t  tissue=palpebral
  locked hyperparameters (from v2_clean_scripts/outputs/swin_t_palpebral_v2_clean/swin_t_palpebral_v2_clean_study_summary.json): lr=0.0314288 wd=4.33528e-06 dropout=0.5

  Structural verification (checkpoints 1-4):
  [PASS] 1. test-set isolation: 0 of 33 held-out test patients appear in any fold
  [PASS] 2. partition integrity: each of 5 repeats partitions all 184 pool patients exactly once
  [PASS] 3. rare-cell coverage: min India_0 per outer val fold = 4 (at r0f0), min Italy_1 per outer val fold = 4 (at r0f0)
  [PASS] 4. pair counts (palpebral): pool n=184 cells={'India_0': 23, 'India_1': 57, 'Italy_0': 84, 'Italy_1': 20} | India 57x23=1311 pairs (32.8x the single split's 40) | Italy 20x84=1680 pairs (28.0x the single split's 60)

  Training 25 fits (5 repeats x 5 folds):
Downloading: "https://download.pytorch.org/models/swin_t-704ceda3.pth" to /root/.cache/torch/hub/checkpoints/swin_t-704ceda3.pt

In [10]:
# Step 1 (transformers) -- 2/6: swin_t_forniceal_palpebral
!python classification/step1_cv_harness/run_cv_harness.py --combo swin_t_forniceal_palpebral_v2_clean
sync_outputs()


Step 1 harness: swin_t_forniceal_palpebral_v2_clean
  device=cuda  arch=swin_t  tissue=forniceal_palpebral
  locked hyperparameters (from v2_clean_scripts/outputs/swin_t_forniceal_palpebral_v2_clean/swin_t_forniceal_palpebral_v2_clean_study_summary.json): lr=0.0626428 wd=3.21806e-05 dropout=0.2

  Structural verification (checkpoints 1-4):
  [PASS] 1. test-set isolation: 0 of 33 held-out test patients appear in any fold
  [PASS] 2. partition integrity: each of 5 repeats partitions all 178 pool patients exactly once
  [PASS] 3. rare-cell coverage: min India_0 per outer val fold = 4 (at r0f0), min Italy_1 per outer val fold = 3 (at r0f4)
  [PASS] 4. pair counts (forniceal_palpebral): pool n=178 cells={'India_0': 23, 'India_1': 57, 'Italy_0': 79, 'Italy_1': 19} | India 57x23=1311 pairs (32.8x the single split's 40) | Italy 19x79=1501 pairs (25.0x the single split's 60)

  Training 25 fits (5 repeats x 5 folds):
    r0f0: stopped epoch 11 (best inner epoch 4, inner_loss=0.6726, train_loss

In [11]:
# Step 1 (transformers) -- 3/6: vit_b_16_palpebral
!python classification/step1_cv_harness/run_cv_harness.py --combo vit_b_16_palpebral_v2_clean
sync_outputs()


Step 1 harness: vit_b_16_palpebral_v2_clean
  device=cuda  arch=vit_b_16  tissue=palpebral
  locked hyperparameters (from v2_clean_scripts/outputs/vit_b_16_palpebral_v2_clean/vit_b_16_palpebral_v2_clean_study_summary.json): lr=0.000293803 wd=2.93754e-06 dropout=0.5

  Structural verification (checkpoints 1-4):
  [PASS] 1. test-set isolation: 0 of 33 held-out test patients appear in any fold
  [PASS] 2. partition integrity: each of 5 repeats partitions all 184 pool patients exactly once
  [PASS] 3. rare-cell coverage: min India_0 per outer val fold = 4 (at r0f0), min Italy_1 per outer val fold = 4 (at r0f0)
  [PASS] 4. pair counts (palpebral): pool n=184 cells={'India_0': 23, 'India_1': 57, 'Italy_0': 84, 'Italy_1': 20} | India 57x23=1311 pairs (32.8x the single split's 40) | Italy 20x84=1680 pairs (28.0x the single split's 60)

  Training 25 fits (5 repeats x 5 folds):
Downloading: "https://download.pytorch.org/models/vit_b_16-c867db91.pth" to /root/.cache/torch/hub/checkpoints/vit_b_

In [12]:
# Step 1 (transformers) -- 4/6: vit_b_16_forniceal_palpebral
!python classification/step1_cv_harness/run_cv_harness.py --combo vit_b_16_forniceal_palpebral_v2_clean
sync_outputs()


Step 1 harness: vit_b_16_forniceal_palpebral_v2_clean
  device=cuda  arch=vit_b_16  tissue=forniceal_palpebral
  locked hyperparameters (from v2_clean_scripts/outputs/vit_b_16_forniceal_palpebral_v2_clean/vit_b_16_forniceal_palpebral_v2_clean_study_summary.json): lr=0.00853372 wd=1.2174e-06 dropout=0.2

  Structural verification (checkpoints 1-4):
  [PASS] 1. test-set isolation: 0 of 33 held-out test patients appear in any fold
  [PASS] 2. partition integrity: each of 5 repeats partitions all 178 pool patients exactly once
  [PASS] 3. rare-cell coverage: min India_0 per outer val fold = 4 (at r0f0), min Italy_1 per outer val fold = 3 (at r0f4)
  [PASS] 4. pair counts (forniceal_palpebral): pool n=178 cells={'India_0': 23, 'India_1': 57, 'Italy_0': 79, 'Italy_1': 19} | India 57x23=1311 pairs (32.8x the single split's 40) | Italy 19x79=1501 pairs (25.0x the single split's 60)

  Training 25 fits (5 repeats x 5 folds):
    r0f0: stopped epoch 23 (best inner epoch 16, inner_loss=0.2962, t

In [13]:
# Step 1 (transformers) -- 5/6: vit_l_16_palpebral
!python classification/step1_cv_harness/run_cv_harness.py --combo vit_l_16_palpebral_v2_clean
sync_outputs()


Step 1 harness: vit_l_16_palpebral_v2_clean
  device=cuda  arch=vit_l_16  tissue=palpebral
  locked hyperparameters (from v2_clean_scripts/outputs/vit_l_16_palpebral_v2_clean/vit_l_16_palpebral_v2_clean_study_summary.json): lr=0.00635836 wd=0.000133112 dropout=0.5

  Structural verification (checkpoints 1-4):
  [PASS] 1. test-set isolation: 0 of 33 held-out test patients appear in any fold
  [PASS] 2. partition integrity: each of 5 repeats partitions all 184 pool patients exactly once
  [PASS] 3. rare-cell coverage: min India_0 per outer val fold = 4 (at r0f0), min Italy_1 per outer val fold = 4 (at r0f0)
  [PASS] 4. pair counts (palpebral): pool n=184 cells={'India_0': 23, 'India_1': 57, 'Italy_0': 84, 'Italy_1': 20} | India 57x23=1311 pairs (32.8x the single split's 40) | Italy 20x84=1680 pairs (28.0x the single split's 60)

  Training 25 fits (5 repeats x 5 folds):
Downloading: "https://download.pytorch.org/models/vit_l_16_lc_swag-4d563306.pth" to /root/.cache/torch/hub/checkpoints

In [14]:
# Step 1 (transformers) -- 6/6: vit_l_16_forniceal_palpebral
!python classification/step1_cv_harness/run_cv_harness.py --combo vit_l_16_forniceal_palpebral_v2_clean
sync_outputs()


Step 1 harness: vit_l_16_forniceal_palpebral_v2_clean
  device=cuda  arch=vit_l_16  tissue=forniceal_palpebral
  locked hyperparameters (from v2_clean_scripts/outputs/vit_l_16_forniceal_palpebral_v2_clean/vit_l_16_forniceal_palpebral_v2_clean_study_summary.json): lr=0.00425263 wd=2.00941e-05 dropout=0.2

  Structural verification (checkpoints 1-4):
  [PASS] 1. test-set isolation: 0 of 33 held-out test patients appear in any fold
  [PASS] 2. partition integrity: each of 5 repeats partitions all 178 pool patients exactly once
  [PASS] 3. rare-cell coverage: min India_0 per outer val fold = 4 (at r0f0), min Italy_1 per outer val fold = 3 (at r0f4)
  [PASS] 4. pair counts (forniceal_palpebral): pool n=178 cells={'India_0': 23, 'India_1': 57, 'Italy_0': 79, 'Italy_1': 19} | India 57x23=1311 pairs (32.8x the single split's 40) | Italy 19x79=1501 pairs (25.0x the single split's 60)

  Training 25 fits (5 repeats x 5 folds):
    r0f0: stopped epoch 22 (best inner epoch 15, inner_loss=0.4205, 

## Checkpoint 5 -- label-shuffle negative control

**Not re-run here.** The negative control tests the harness's fold-construction and pooling logic
for leakage -- that logic is identical regardless of which architecture is trained through it, and
it was already run and passed in `step1-cv-harness-cnn.ipynb` (`mobilenet_v3_small_palpebral_v2_clean`,
both `within_country` and `global` modes). Re-running it here would test the same thing a second time
at real compute cost, not add new evidence. If you want to re-verify it against a transformer
specifically, run:

```
!python classification/step1_cv_harness/run_cv_harness.py \
    --combo swin_t_palpebral_v2_clean --shuffle-control within_country
```

## Aggregate -- checkpoints 7, 8, 9

Writes a Step 1 baseline from whatever combos are present in `outputs/` and evaluates the
remaining gates. Exits non-zero and refuses to declare Step 1 clear if any gate fails.

- **7 plausibility** -- pooled India AUC far outside what the single-split CIs already permitted
  means suspect the harness, not celebrate a discovery.
- **8 precision** -- India AUC 95% CI half-width <= 0.12. The actual pass/fail criterion.
- **9 artifact** -- the baseline file itself, recording seed, fold config and locked
  hyperparameters so later steps can run a valid *paired* comparison against it.

**Note: run standalone, this aggregates only the 6 transformer combos present in `outputs/` at
this point** -- gate 5 (negative control) will report `n_controls_run: 0` and fail here unless you
ran the optional cell above, since no control was produced in this notebook by default (see note
above -- it was already run in the CNN notebook). That does not mean this notebook's numbers are
untrustworthy; it means the combined **18-combo** baseline -- built by merging this notebook's
and `step1-cv-harness-cnn.ipynb`'s output zips locally and re-running `aggregate_baseline.py` once
over both -- is the one that will show gate 5 passing, since it picks up the CNN notebook's control
run too. Combo discovery is dynamic (globs `outputs/*/cv_metrics.json`), so no code change is needed.

In [15]:
!python classification/step1_cv_harness/aggregate_baseline.py
sync_outputs()


Step 1 baseline -- 6 combos
                                combo  india_auc  india_ci_half_width  italy_auc       gap
  swin_t_forniceal_palpebral_v2_clean   0.647597             0.142649   0.882745 -0.235148
            swin_t_palpebral_v2_clean   0.633715             0.145309   0.820238 -0.186523
vit_b_16_forniceal_palpebral_v2_clean   0.680244             0.140351   0.842239 -0.161994
          vit_b_16_palpebral_v2_clean   0.739893             0.125114   0.846071 -0.106178
vit_l_16_forniceal_palpebral_v2_clean   0.698551             0.129300   0.912458 -0.213908
          vit_l_16_palpebral_v2_clean   0.683143             0.136947   0.857262 -0.174119

Gates:
  [FAIL] 5_negative_control
         -> run: run_cv_harness.py --combo <name> --shuffle-control within_country
  [PASS] 7_plausibility
  [FAIL] 8_precision

Wrote /kaggle/working/eyes-defy-anemia/classification/step1_cv_harness/outputs/baseline/step1_baseline.csv
      /kaggle/working/eyes-defy-anemia/classification/step1_cv

In [16]:
from pathlib import Path

report = Path('classification/step1_cv_harness/outputs/baseline/step1_baseline.md')
print(report.read_text(encoding='utf-8') if report.exists() else 'Baseline not produced -- check the aggregate cell above.')

# Step 1 Baseline -- Pooled Out-of-Fold Repeated Cross-Validation

Generated: 2026-08-08T20:26:51.432932+00:00

Frozen reference for Steps 2-5. Every later intervention must be compared against this
using a **paired** bootstrap on the difference (`cv_stats.paired_delta_auc`) with the
identical fold assignments recorded in each combo's `fold_manifest.json` -- not by
checking whether two independent confidence intervals overlap.

## Configuration

- Design: 5-fold x 5 repeats, stratified on country x label
- Seed: 42
- Bootstrap replicates: 2000
- Pool: train + val only; the 33-patient test split is sealed for Step 6
- Hyperparameters: each combo's own winning Optuna trial, reused verbatim (no re-tuning)

## Precision achieved

| | India pairs | Italy pairs |
|---|---|---|
| Single 70/15/15 split | 40 | 60 |
| Pooled out-of-fold | 1311 | 1680 |

## Results (sorted by India AUC)

| Combo | India AUC [95% CI] | Italy AUC [95% CI] | Overall AUC | Gap [95% CI] | Gap excl. 0 |
|---|---|---|--

## Done -- what to download

`/kaggle/working/step1_cv_results_vit.zip` contains, per combo:
- `oof_predictions.csv` -- one held-out probability per patient per repeat
- `fold_manifest.json` -- the exact fold assignments -- Steps 3-5 **must** reuse these so their
  comparison against the baseline can be paired
- `cv_metrics.json` -- pooled AUC + bootstrap CIs, gate results
- `fold_metrics.csv` -- per-fold (own held-out set) full metric set, one row per (repeat, fold)
- `fold_diagnostics.json` -- raw per-fold loss histories + confusion matrices + ROC curves
- `plots/{combo}_loss_curves_grid.png`, `_confusion_matrices_grid.png`, `_roc_curves_grid.png`,
  `_fold_metrics_summary.png` -- one combined image per plot type per combo, all folds side by side

Plus this notebook's own partial `baseline/`, `comparison/`, and `structural_verification.json`.

Still no model checkpoints, by design.

**Next step after both notebooks finish:** download this zip and `step1-cv-harness-cnn.ipynb`'s
zip, extract both `outputs/` into the same local `classification/step1_cv_harness/outputs/`, then
run `aggregate_baseline.py` once locally for the combined 18-combo baseline.

In [17]:
from pathlib import Path

print('Final contents of /kaggle/working/outputs:')
for f in sorted(Path('/kaggle/working/outputs').rglob('*')):
    if f.is_file():
        print(f'  {f.relative_to("/kaggle/working/outputs")}  ({f.stat().st_size / 1e6:.3f} MB)')

zip_path = Path('/kaggle/working/step1_cv_results_vit.zip')
print(f'\nZip archive: {zip_path}  ({zip_path.stat().st_size / 1e6:.2f} MB)')

Final contents of /kaggle/working/outputs:
  baseline/step1_baseline.csv  (0.003 MB)
  baseline/step1_baseline.json  (0.007 MB)
  baseline/step1_baseline.md  (0.003 MB)
  comparison/country_auc_comparison.png  (0.065 MB)
  comparison/india_italy_gap_comparison.png  (0.052 MB)
  structural_verification.json  (0.003 MB)
  swin_t_forniceal_palpebral_v2_clean/cv_metrics.json  (0.008 MB)
  swin_t_forniceal_palpebral_v2_clean/fold_diagnostics.json  (0.050 MB)
  swin_t_forniceal_palpebral_v2_clean/fold_manifest.json  (0.175 MB)
  swin_t_forniceal_palpebral_v2_clean/fold_metrics.csv  (0.003 MB)
  swin_t_forniceal_palpebral_v2_clean/oof_predictions.csv  (0.039 MB)
  swin_t_forniceal_palpebral_v2_clean/plots/swin_t_forniceal_palpebral_v2_clean_confusion_matrices_grid.png  (0.089 MB)
  swin_t_forniceal_palpebral_v2_clean/plots/swin_t_forniceal_palpebral_v2_clean_fold_metrics_summary.png  (0.112 MB)
  swin_t_forniceal_palpebral_v2_clean/plots/swin_t_forniceal_palpebral_v2_clean_loss_curves_grid.pn